# MultiOmicBipartiteGNN

In [2]:
import numpy as np
import polars as pl
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import optuna
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
import matplotlib.pyplot as plt
import torch
from torch_geometric.data import Data, HeteroData
import networkx as nx
from torch_geometric.utils import to_networkx
import torch
from torch.nn import Linear
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATv2Conv, to_hetero
from sklearn.manifold import TSNE
import lightning as L
from lightning.pytorch.callbacks.early_stopping import EarlyStopping
from torch.utils.data import DataLoader
from torchvision import transforms
import torch_geometric.transforms as T
import torch_geometric

# loading data

In [23]:
rna = pl.read_csv("xena_brca_data/TCGA.BRCA.sampleMap_HiSeqV2", separator="\t")

gene_names = rna[:,0]
# rna = rna.drop("sample")

# load labels & filter rows with nulls
clinicalMatrix = pl.read_csv("xena_brca_data/TCGA.BRCA.sampleMap_BRCA_clinicalMatrix", separator="\t", infer_schema_length=0)
cM = clinicalMatrix.filter(pl.col("PAM50_mRNA_nature2012") != "null")

labels = cM.select(["sampleID","PAM50_mRNA_nature2012"])
labels = labels.filter(pl.col("PAM50_mRNA_nature2012") != "Normal-like")

y_all = labels["PAM50_mRNA_nature2012"].to_numpy()
np.unique(labels["PAM50_mRNA_nature2012"].to_numpy(), return_counts=True)

(array(['Basal-like', 'HER2-enriched', 'Luminal A', 'Luminal B'],
       dtype=object),
 array([ 98,  58, 231, 127]))

In [37]:
cnv = pl.read_csv("xena_brca_data/Gistic2_CopyNumber_Gistic2_all_data_by_genes", separator="\t")
cnvth = pl.read_csv("xena_brca_data/Gistic2_CopyNumber_Gistic2_all_thresholded.by_genes", separator="\t")

In [7]:
mirna1 = pl.read_csv("xena_brca_data/TCGA.BRCA.sampleMap_miRNA_HiSeq_gene", separator="\t", null_values=["NA"])
mirna2 = pl.read_csv("xena_brca_data/miRNA_GA_gene", separator="\t", null_values=["NA"])
# mirna.select(labels["sampleID"].to_list())

In [25]:
samples_name_list = labels['sampleID'].to_list()
samples_name_set = set(samples_name_list)

mirna_names_set = set(mirna1.columns[1:] + mirna2.columns[1:])

cnv_set = set(cnv.columns[1:])

rna_names_set = set(rna.columns[1:])

common_names = samples_name_set & mirna_names_set & cnv_set & rna_names_set

len(common_names)

sample_names = list(common_names)

In [30]:
sample_names.sort()

In [44]:
sample_names

['TCGA-A1-A0SD-01',
 'TCGA-A1-A0SE-01',
 'TCGA-A1-A0SH-01',
 'TCGA-A1-A0SJ-01',
 'TCGA-A1-A0SK-01',
 'TCGA-A1-A0SM-01',
 'TCGA-A1-A0SO-01',
 'TCGA-A2-A04N-01',
 'TCGA-A2-A04P-01',
 'TCGA-A2-A04Q-01',
 'TCGA-A2-A04R-01',
 'TCGA-A2-A04T-01',
 'TCGA-A2-A04U-01',
 'TCGA-A2-A04V-01',
 'TCGA-A2-A04W-01',
 'TCGA-A2-A04X-01',
 'TCGA-A2-A04Y-01',
 'TCGA-A2-A0CL-01',
 'TCGA-A2-A0CM-01',
 'TCGA-A2-A0CP-01',
 'TCGA-A2-A0CQ-01',
 'TCGA-A2-A0CS-01',
 'TCGA-A2-A0CT-01',
 'TCGA-A2-A0CU-01',
 'TCGA-A2-A0CV-01',
 'TCGA-A2-A0CW-01',
 'TCGA-A2-A0D0-01',
 'TCGA-A2-A0D1-01',
 'TCGA-A2-A0D2-01',
 'TCGA-A2-A0D3-01',
 'TCGA-A2-A0D4-01',
 'TCGA-A2-A0EM-01',
 'TCGA-A2-A0EN-01',
 'TCGA-A2-A0EO-01',
 'TCGA-A2-A0EQ-01',
 'TCGA-A2-A0ER-01',
 'TCGA-A2-A0ES-01',
 'TCGA-A2-A0ET-01',
 'TCGA-A2-A0EU-01',
 'TCGA-A2-A0EV-01',
 'TCGA-A2-A0EW-01',
 'TCGA-A2-A0EX-01',
 'TCGA-A2-A0EY-01',
 'TCGA-A2-A0ST-01',
 'TCGA-A2-A0SU-01',
 'TCGA-A2-A0SV-01',
 'TCGA-A2-A0SW-01',
 'TCGA-A2-A0SX-01',
 'TCGA-A2-A0SY-01',
 'TCGA-A2-A0T0-01',


In [34]:
cM_common_samples = cM.filter(pl.col("sampleID").is_in(sample_names))
cM_sl = cM_common_samples.select(["sampleID", "PAM50_mRNA_nature2012"])
cM_sl.write_csv("BRCA_PROCESSED_DATA/labels.tsv", separator="\t")
# cM_sl.sort(pl.col("sampleID"))

sampleID,PAM50_mRNA_nature2012
str,str
"""TCGA-A1-A0SD-0…","""Luminal A"""
"""TCGA-A1-A0SE-0…","""Luminal A"""
"""TCGA-A1-A0SH-0…","""Luminal A"""
"""TCGA-A1-A0SJ-0…","""Luminal A"""
"""TCGA-A1-A0SK-0…","""Basal-like"""
…,…
"""TCGA-E2-A1B4-0…","""Luminal A"""
"""TCGA-E2-A1B5-0…","""Basal-like"""
"""TCGA-E2-A1B6-0…","""Luminal A"""


In [250]:
# select and preprocess mirnas
mirna1_names = mirna["sample"].to_list()
mirna2_names = mirna2["sample"].to_list()

mirna_names = list(set(mirna1_names) & set(mirna2_names))
mirna1 = mirna.filter(pl.col("sample").is_in(mirna_names))
mirna2 = mirna2.filter(pl.col("sample").is_in(mirna_names))
mirna = mirna1.join(mirna2, on=["sample"])
mirna_c = mirna.select(["sample"] + sample_names)
mirna_c = mirna_c.drop_nulls()
mirna_c.write_csv("BRCA_PROCESSED_DATA/mirna.tsv", separator="\t")

In [251]:
# select and preprocess mrna
mrna_genes = rna[:,0]
mrna = rna.select(["sample"] + sample_names)

# drop low var features
mrna_mat = mrna[:,1:].to_numpy()
mrna_vars = mrna_mat.var(axis=1)
high_var_mask = mrna_vars > 0.1
high_var_idx = np.where(high_var_mask)[0]

mrna = mrna[high_var_idx]

mrna.write_csv("BRCA_PROCESSED_DATA/mrna.tsv", separator="\t")

In [255]:
cnv = cnv_hv_df.filter(pl.col("Gene Symbol").is_in(mrna["sample"]))

In [256]:
cnv.write_csv("BRCA_PROCESSED_DATA/cnv.tsv", separator="\t")

In [46]:
len(sample_names)

483

In [57]:
# select and preprocess cnv
cnvs = cnv.select(["Gene Symbol"] + sample_names)
cnvths = cnvth.select(["Gene Symbol"] + sample_names)

# cnv = cnv.filter(pl.col("Gene Symbol").is_in(mrna["sample"]))
# cnvth = cnvth.filter(pl.col("Gene Symbol").is_in(mrna["sample"]))

# drop low var features
cnv_mat = cnvs[:,1:].to_numpy()
cnv_vars = cnv_mat.var(axis=1)
high_var_mask = cnv_vars > 0.1
high_var_idx = np.where(high_var_mask)[0]

cnv_sel = cnvs[high_var_idx]


# drop low var features
cnvth_mat = cnvths[:,1:].to_numpy()
cnvth_vars = cnvth_mat.var(axis=1)
high_var_mask = cnvth_vars > 0.1
high_var_idx = np.where(high_var_mask)[0]

# filter by variance
cnvth_sel = cnvths[high_var_idx]

print(cnv_sel)
print(cnvth_sel)

# cnv.write_csv("BRCA_PROCESSED_DATA/cnv.tsv", separator="\t")
# cnvth.write_csv("BRCA_PROCESSED_DATA/cnvth.tsv", separator="\t")

shape: (16_711, 484)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ Gene      ┆ TCGA-A1-A ┆ TCGA-A1-A ┆ TCGA-A1-A ┆ … ┆ TCGA-E2-A ┆ TCGA-E2-A ┆ TCGA-E2-A ┆ TCGA-E2- │
│ Symbol    ┆ 0SD-01    ┆ 0SE-01    ┆ 0SH-01    ┆   ┆ 1B5-01    ┆ 1B6-01    ┆ 1BC-01    ┆ A1BD-01  │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│ str       ┆ f64       ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ AJAP1     ┆ -0.521    ┆ -0.06     ┆ -0.398    ┆ … ┆ 0.031     ┆ -0.011    ┆ -0.006    ┆ 0.005    │
│ CAMTA1    ┆ -0.521    ┆ -0.06     ┆ -0.412    ┆ … ┆ 0.031     ┆ -0.011    ┆ -0.006    ┆ 0.005    │
│ RN7SL729P ┆ -0.521    ┆ -0.06     ┆ -0.412    ┆ … ┆ 0.031     ┆ -0.011    ┆ -0.006    ┆ 0.005    │
│ SLC45A1   ┆ -0.521    ┆ -0.06     ┆ -0.412    ┆ … ┆ 0.031     ┆ -0.0

In [261]:
meth = pl.read_csv("xena_brca_data/TCGA.BRCA.sampleMap_HumanMethylation450", separator="\t", null_values=["NA"])
meth = meth.drop_nulls()

In [269]:
meth27 = pl.read_csv("xena_brca_data/HumanMethylation27", separator="\t", null_values=["NA"])
meth27 = meth27.drop_nulls()

In [272]:
len(set(meth.columns + meth27.columns) & set(sample_names))

483

In [191]:
cnvmat_vars = cnvmat.var(axis=1)
high_var_mask = cnvmat_vars > 0.1
high_var_idx = np.where(high_var_mask)[0]

# Building input graph

In [ ]:
def create_expression_connections(X, std_multiplier=1.0):
    """
    This function identifies and categorizes gene expression levels as 
    under-expressed (-1), over-expressed (1), or baseline (0) based on standard 
    deviation from the mean.
    
    Args:
     X (np.ndarray): A 2D NumPy array representing gene expression data, 
         with shape (samples, genes).
     std_multiplier (float, optional): A hyperparameter that scales the 
         standard deviation to define the expression bounds. Higher values will result
         in stricter bounds (std_multiplier=2.0 will consider values further than 2 sigma
         from the mean to be differential) Defaults to 1.0.
    
    Returns:
     np.ndarray: A 2D NumPy array with the same shape as X, where each element 
         indicates the expression category (-1, 0, or 1) for the corresponding 
         gene in each sample.
    """
    mean_exps = X.mean(axis=0)
    exps_std = X.std(axis=0)
    
    lb_exps = mean_exps - exps_std * std_multiplier
    ub_exps = mean_exps + exps_std * std_multiplier
    
    A_exps = np.zeros_like(X)

    mask_below = X <= lb_exps
    mask_above = X >= ub_exps
    
    A_exps[mask_below] = -1  # Set under-expressed elements
    A_exps[mask_above] = 1  # Set over-expressed elements

    return A_exps

def cosine_similarity_matrix(matrix):
    """
    given a matrix of (n_samples, n_features) compute the cosine similarities, between the samples
    """
    # Compute dot product between all pairs of vectors
    dot_products = np.dot(matrix, matrix.T)
    
    # Compute magnitudes of all vectors
    magnitudes = np.linalg.norm(matrix, axis=1)
    
    # Compute outer product of magnitudes to obtain matrix of magnitudes product
    magnitude_products = np.outer(magnitudes, magnitudes)
    
    # Compute cosine similarity matrix
    cosine_similarities = dot_products / magnitude_products
    
    return cosine_similarities

def keep_n_neighbours(A, n):
      rows, cols = A.shape
      for i in range(rows):
        # Find indices of the k highest elements
        bottom_k_indices = np.argsort(A[i])[:-n]
        # Set all other elements to zero
        A[i][bottom_k_indices] = 0
      return A
    

def dense_to_coo(adj_mat):
    """
    convert an adjacency matrix in a dense format to a torch tensor in COO format
    """
    indices = np.where(adj_mat != 0)
    indices = np.vstack((indices[0], indices[1]))
    return torch.tensor(indices, dtype=torch.int64)
    
def dense_to_attributes(adj_mat):
    """
    return the flattened weight entries from the matrix
    for use as edge attributes
    """
    return torch.tensor(adj_mat[adj_mat != 0]).view(-1,1)

In [ ]:
data = HeteroData()
data['mrna_samples'].x=Xt
data['cnv_samples'].x=
data['mirna_samples'].x=
data.y=y
data.train_mask=train_mask
data.val_mask=val_mask
data.test_mask=test_mask
data['mrna_genes'].x=torch.ones(Xt.shape[1], Xt.shape[1]) # <- set initial gene features to ones
data['cnv_genes'].x=

data['mrna', 'connected_to', 'feature'].edge_index=sample_to_gene_connections
data['sample', 'connected_to', 'feature'].edge_attributes=dense_to_attributes(A_exps)
data['feature', 'interacts_with', 'feature'].edge_index=interaction_connections
data['sample', 'similar_to', 'sample'].edge_index=dense_to_coo(torch.eye(Xt.shape[0]))

In [279]:
a = np.random.randint(low=0, high=10, size=(5,5))
print(a)
b = np.eye(5)
b[0,0]=2
print(b)

b @ a

[[1 6 2 2 3]
 [2 5 0 9 1]
 [8 9 3 5 6]
 [1 0 9 1 1]
 [9 9 1 3 3]]
[[2. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1.]]


array([[ 2., 12.,  4.,  4.,  6.],
       [ 2.,  5.,  0.,  9.,  1.],
       [ 8.,  9.,  3.,  5.,  6.],
       [ 1.,  0.,  9.,  1.,  1.],
       [ 9.,  9.,  1.,  3.,  3.]])

In [ ]:
class RBiGAT(torch.nn.Module):
    def __init__(self, hidden_channels, out_channels, num_layers, proj_dim=100):
        super().__init__()
        
        self.convs = torch.nn.ModuleList()

        self.mrna_proj = Linear(data['mrna_samples'].shape[1], proj_dim)
        self.cnv_proj = Linear(data['cnv_samples'].shape[1], proj_dim)
        self.mirna_proj = Linear(data['mirna_samples'].shape[1], proj_dim)

        # TODO - this could be improved
        for _ in range(num_layers):
            conv = HeteroConv({
                ('mrna_samples', 'connected_to', 'mrna_features'): GATv2Conv((-1, -1), hidden_channels, dropout=0.85, add_self_loops=False),
                ('mrna_samples', 'rev_connected_to', 'mrna_features'): GATv2Conv((-1, -1), hidden_channels, dropout=0.85, add_self_loops=False),
                
                ('cnv_samples', 'connected_to', 'cnv_features'): GATv2Conv((-1, -1), hidden_channels, dropout=0.85, add_self_loops=False),
                ('cnv_samples', 'rev_connected_to', 'cnv_features'): GATv2Conv((-1, -1), hidden_channels, dropout=0.85, add_self_loops=False),
                
                ('mirna_samples', 'connected_to', 'mirna_features'): GATv2Conv((-1, -1), hidden_channels, dropout=0.85, add_self_loops=False),
                ('mirna_samples', 'rev_connected_to', 'mirna_features'): GATv2Conv((-1, -1), hidden_channels, dropout=0.85, add_self_loops=False),
                
                ('mrna_samples', 'interacts_with', 'mrna_features'): GATv2Conv((-1, -1), hidden_channels, dropout=0.85, add_self_loops=False),
                ('cnv_samples', 'interacts_with', 'cnv_features'): GATv2Conv((-1, -1), hidden_channels, dropout=0.85, add_self_loops=False),
                
                ('sample', 'similar_to', 'sample'): GCNConv(-1, hidden_channels),
            }, aggr='sum')
            self.convs.append(conv)

        # prediction integration layer
        self.lin = Linear(
            hidden_channels * 3,
            out_channels
        )

    def forward(self, x_dict, edge_index_dict):

        # apply projections
        x_dict['mrna_samples'] = self.mrna_proj(x_dict['mrna_samples'])
        x_dict['cnv_samples'] = self.cnv_proj(x_dict['cnv_samples'])
        x_dict['mirna_samples'] = self.mirna_proj(x_dict['mirna_samples'])
        
        for conv in self.convs:
            x_dict = conv(x_dict, edge_index_dict)
            x_dict = {key: x.relu() for key, x in x_dict.items()}

        # stack prediction from each sample vertex and integrate them with a projection module
        out_stack = torch.stack(
            [x_dict['mrna_samples'], x_dict['cnv_samples'], x_dict['mirna_samples']],
            dim=0,
        )
        
        return self.lin(out_stack)   